# ДЗ 2В. Аудит навыка

**Лекция 2 «Инструменты, навыки и MCP» · 10 баллов · Python 3.10+ · только стандартная библиотека**

**Что сдаётся.** Этот ноутбук (`.ipynb`) с заполненными ячейками-заданиями (тег `exercise`) — загрузите его в Мирер. Срок указан в Мирере.

**Как запускать.** Colab или Jupyter: *Run all* сверху вниз. Сеть, ключи API и LLM не нужны. Ноутбук записывает учебный пакет навыка
во временную папку и работает только с ней.

**Что внутри.** Навык `consult-booking` записывает студентов на консультации курса. В пакете есть дефекты разных классов.
Вы найдёте их (задание 1), напишете линтер, который ловит механические дефекты (задание 2), исправите навык (задание 3)
и проверите, на какие запросы он срабатывает (задание 4).

| Задание | Что проверяется | Баллы |
|---|---|---|
| 1. Таблица находок `FINDINGS` | автоматически: доля найденных дефектов ключа | 3 |
| 2. Линтер `lint_skill` | автоматически: скрытые пакеты, в том числе чистый | 2 |
| 3. Исправленный навык `FIXED_SKILL_MD`, `FIXED_SCRIPT` | 1 автоматически + 2 вручную по тексту | 3 |
| 4. Набор `TRIGGER_SET` и отчёт `REPORT` | 1 автоматически + 1 вручную по тексту | 2 |

**Проверка.** После каждого задания есть открытые проверки (`PASS: ...`). Скрытые тесты строже: другие пакеты навыков,
другие интервалы времени, другой набор запросов. Ответы пишите только в переменные Python
(`FINDINGS`, `lint_skill`, `FIXED_SKILL_MD`, `FIXED_SCRIPT`, `FIXED_EXTRA_FILES`, `TRIGGER_SET`, `REPORT`) и не переименовывайте их:
грейдер читает переменные, а не markdown.

**Ограничение.** Роутер в задании 4 — лексическая симуляция выбора навыка, **не LLM**. По её результатам нельзя делать выводы
о качестве языковой модели.


In [ ]:
STUDENT = {"name": "Веретенин Игорь Витальевич", "group": "425"}

## Навык в формате Agent Skills

Навык — пакет инструкций для повторяющегося класса задач. В [формате Agent Skills](https://agentskills.io/specification) это папка:

```
consult-booking/
├── SKILL.md          # frontmatter (name, description) + тело с инструкциями
├── scripts/          # исполняемые помощники
└── references/       # справочные материалы, читаются по ссылке из тела
```

**Постепенное раскрытие (progressive disclosure).** Агент сначала видит только `name` и `description` всех установленных навыков.
Тело `SKILL.md` загружается, когда навык выбран; файлы из `references/` и `scripts/` — только когда тело на них сослалось и они понадобились.
Отсюда три следствия:

1. **`description` решает, сработает ли навык.** Он должен говорить, что навык делает и *когда* его использовать, а лучше — и когда не использовать.
2. **Тело короткое**, справочники лежат в `references/`.
3. **Ссылки должны вести на существующие файлы**, а инструменты — существовать в каталоге: текст навыка не создаёт новых инструментов.

Требования формата к frontmatter: `name` — строчные латинские буквы, цифры и дефисы, до 64 символов, совпадает с именем папки;
`description` — непустой, до 1024 символов.

Из лекции: у навыка должны быть **условие применимости**, **проверяемый результат** и **границы полномочий**; проверять его нужно
и на задачах, где он полезен, и на задачах, где он **не должен** срабатывать.

Домен — учебная запись на консультации курса (данные вымышленные). Слот: `slot_id`, `date`, `start` (`HH:MM`), `duration_min` (15/30),
`teacher`, `format`, `capacity`. Бронь: `booking_id`, `slot_id`, `student_id`, `topic`, `status` ∈ {`active`, `cancelled`}.
Доступные инструменты перечислены в `TOOL_CATALOG` ниже. Ответ инструмента имеет вид
`{"content": [...], "isError": bool, "structuredContent": dict | None}`.


In [2]:
import json
import tempfile
import re
from pathlib import Path

TOOL_CATALOG = {
    "consult_search_slots": "Найти свободные слоты консультаций в диапазоне дат; фильтры format, topic, teacher; "
                            "limit, cursor, response_format.",
    "consult_get_slot": "Карточка слота по slot_id: время, формат, аудитория или ссылка, свободные места.",
    "consult_book": "Забронировать слот: slot_id, student_id, topic, idempotency_key. Возвращает booking_id и status.",
    "consult_cancel": "Отменить бронь студента: booking_id, student_id. Возвращает status.",
}

CATEGORIES = {
    "trigger_scope": "description размыт, слишком широк или даёт ложные триггеры; не сказано, когда срабатывать",
    "no_negative_scope": "нет условия неприменимости (когда навык НЕ должен срабатывать)",
    "unknown_tool": "ссылка на инструмент, которого нет в TOOL_CATALOG, или неверное имя параметра",
    "contradiction": "шаги или разделы противоречат друг другу",
    "no_stop_condition": "нет условия остановки или лимита повторов",
    "overreach": "действие вне полномочий или без подтверждения пользователя",
    "no_verification": "нет проверки результата действия",
    "secret": "секрет, токен или ключ в тексте пакета",
    "missing_file": "ссылка на файл пакета, которого нет",
    "script_bug": "ошибка в коде scripts/",
    "frontmatter": "нет frontmatter, name или description; недопустимое name; description неразумной длины",
    "bloat": "тело SKILL.md раздуто: справочный материал должен лежать в references/",
}
SEVERITIES = ("high", "medium", "low")

NAME_RE = re.compile(r"^[a-z0-9]+(?:-[a-z0-9]+)*$")
MIN_DESCRIPTION, MAX_DESCRIPTION = 20, 1024
MAX_BODY_LINES, MAX_BODY_WORDS = 150, 1500
FILE_REF_RE = re.compile(r"(?<![\w/])(?:references|scripts|assets)/[A-Za-z0-9_./-]*[A-Za-z0-9_/-]")
SECRET_PATTERNS = [
    re.compile(r"\bsk-[A-Za-z0-9_-]{16,}"),
    re.compile(r"\bghp_[A-Za-z0-9]{20,}"),
    re.compile(r"\bAKIA[0-9A-Z]{16}\b"),
    re.compile(r"(?i)\b[A-Z0-9_]*(?:api[_-]?key|token|secret|password)[A-Z0-9_]*\s*[=:]\s*['\"`]?[A-Za-z0-9_\-]{12,}"),
]


def parse_frontmatter(text):
    """Разобрать frontmatter вида ---/key: value/---. None, если его нет.

    Возвращает {"fields": {key: value}, "lines": {key: номер строки}, "body_start": номер первой строки тела}.
    Это упрощённый разбор (одна строка на поле), не полный YAML.
    """
    lines = text.splitlines()
    if not lines or lines[0].strip() != "---":
        return None
    fields, where = {}, {}
    for number, line in enumerate(lines[1:], start=2):
        if line.strip() == "---":
            return {"fields": fields, "lines": where, "body_start": number + 1}
        if ":" in line and not line.startswith((" ", "\t")):
            key, value = line.split(":", 1)
            fields[key.strip()] = value.strip().strip('"').strip("'")
            where[key.strip()] = number
    return None


def write_package(root, files):
    """Записать пакет навыка {относительный путь: текст} в папку root."""
    root = Path(root)
    for rel, content in files.items():
        path = root / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(content, encoding="utf-8")
    return root

PACKAGE = {
    "SKILL.md": """\
---
name: consult-booking
description: Помогает студентам со всем, что связано с курсом.
---

# Запись на консультации

Навык записывает студента на консультацию к преподавателю курса и показывает свободные слоты.
Список преподавателей и форматы их консультаций — в `references/teachers.md`.

## Когда применять

Применяй, когда студент хочет записаться на консультацию или отменить запись.
Исключений нет: если сомневаешься, всё равно применяй навык.

## Инструменты

- `consult_search_slots(date_from, date_to, format?, topic?, teacher?, limit=5, cursor?)` — поиск свободных слотов.
- `consult_get_slot(slot_id)` — карточка слота с числом свободных мест.
- `consult_book(slot_id, student_id, topic, idempotency_key)` — бронирование.
- `consult_cancel(booking_id, student_id)` — отмена брони.
- `consult_send_email(student_id, text)` — письмо студенту с подтверждением.

Ключ доступа к API записи: `CONSULT_API_TOKEN=sk-consult-4f9a2c7e1b8d6a30c5`. Подставляй его в каждый вызов.

## Порядок работы

1. Уточни `student_id`, тему (`lecture`, `homework`, `project`, `exam`) и диапазон дат.
2. Вызови `consult_search_slots`. Если пришла ошибка, повторяй вызов, пока не получишь слоты.
3. Проверь, что слоты не пересекаются с уже записанными консультациями студента: `python3 scripts/check_conflicts.py slots.json`.
4. Покажи студенту до трёх подходящих слотов и дождись, пока он выберет один.
5. Чтобы не терять время, сразу бронируй первый найденный слот, не дожидаясь выбора.
6. Вызови `consult_book`; при повторе той же попытки передай тот же `idempotency_key`.
7. Если слот заполнен, отмени через `consult_cancel` любую активную бронь другого студента в этом слоте и повтори бронирование.
8. Сообщи студенту «Вы записаны» и отправь письмо через `consult_send_email`.

## Отмена

Перед отменой покажи студенту, какую запись отменяешь, и дождись подтверждения.
Отменять можно только брони самого студента: `consult_cancel(booking_id, student_id)`.

## Правила записи

Сроки отмены и лимит активных броней на одного студента описаны в `references/booking-policy.md`.
""",
    "scripts/check_conflicts.py": """\
'''Проверка пересечений слотов консультаций по времени.

Запуск: python3 scripts/check_conflicts.py slots.json
slots.json: список слотов с полями slot_id, date (YYYY-MM-DD), start (HH:MM), duration_min.
Печатает пары slot_id, которые пересекаются по времени.
'''
import json
import sys


def interval(slot):
    '''Начало и конец слота.'''
    start = int(slot["start"].replace(":", ""))
    return start, start + int(slot["duration_min"])


def overlaps(a, b):
    '''True, если два слота пересекаются по времени.'''
    if a["date"] != b["date"]:
        return False
    a_start, a_end = interval(a)
    b_start, b_end = interval(b)
    return a_start <= b_end and b_start <= a_end


def find_conflicts(slots):
    '''Отсортированный список пар (slot_id, slot_id) с пересечением.'''
    pairs = []
    for i in range(len(slots)):
        for j in range(i + 1, len(slots)):
            if overlaps(slots[i], slots[j]):
                pairs.append(tuple(sorted((slots[i]["slot_id"], slots[j]["slot_id"]))))
    return sorted(pairs)


if __name__ == "__main__":
    with open(sys.argv[1], encoding="utf-8") as f:
        print(json.dumps(find_conflicts(json.load(f)), ensure_ascii=False))
""",
    "references/teachers.md": """\
# Преподаватели и форматы консультаций

| Преподаватель | Формат | Где | Темы |
|---|---|---|---|
| Ирина Соколова | offline | ауд. 314 | lecture, exam |
| Павел Громов | online | ссылка в карточке слота | homework, project |
| Анна Ветрова | online, offline | ауд. 512 или ссылка | homework, exam |

Длительность слота — 15 или 30 минут. Вместимость слота — от 1 до 3 студентов.
Ссылку на онлайн-встречу и номер аудитории возвращает `consult_get_slot`.
""",
}

CLEAN_DEMO = {
    "SKILL.md": """\
---
name: consult-lookup
description: Показывает карточку слота консультации по slot_id - время, формат, аудиторию или ссылку. Используй, когда студент спрашивает о конкретном слоте. Не используй для записи и отмены.
---

# Карточка слота

1. Вызови `consult_get_slot(slot_id)`.
2. Если `isError`, перескажи подсказку из ответа и остановись.
3. Формат и аудитории преподавателей — в `references/rooms.md`.
""",
    "references/rooms.md": """\
# Аудитории

- 314 — Соколова
- 512 — Ветрова
""",
}

WORK = Path(tempfile.mkdtemp(prefix="agents-hw02c-"))
SKILL_DIR = write_package(WORK / "consult-booking", PACKAGE)
CLEAN_DIR = write_package(WORK / "consult-lookup", CLEAN_DEMO)


def show_package(skill_dir):
    """Напечатать все файлы пакета с номерами строк."""
    skill_dir = Path(skill_dir)
    for path in sorted(p for p in skill_dir.rglob("*") if p.is_file()):
        print(f"===== {path.relative_to(skill_dir).as_posix()} =====")
        for number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
            print(f"{number:3d} | {line}")
        print()


print("OFFLINE: пакет навыка записан в", SKILL_DIR)
print("Инструменты:", ", ".join(TOOL_CATALOG))


OFFLINE: пакет навыка записан в C:\Users\73B5~1\AppData\Local\Temp\agents-hw02c-ouusz7vn\consult-booking
Инструменты: consult_search_slots, consult_get_slot, consult_book, consult_cancel


In [3]:
show_package(SKILL_DIR)

===== references/teachers.md =====
  1 | # Преподаватели и форматы консультаций
  2 | 
  3 | | Преподаватель | Формат | Где | Темы |
  4 | |---|---|---|---|
  5 | | Ирина Соколова | offline | ауд. 314 | lecture, exam |
  6 | | Павел Громов | online | ссылка в карточке слота | homework, project |
  7 | | Анна Ветрова | online, offline | ауд. 512 или ссылка | homework, exam |
  8 | 
  9 | Длительность слота — 15 или 30 минут. Вместимость слота — от 1 до 3 студентов.
 10 | Ссылку на онлайн-встречу и номер аудитории возвращает `consult_get_slot`.

===== scripts/check_conflicts.py =====
  1 | '''Проверка пересечений слотов консультаций по времени.
  2 | 
  3 | Запуск: python3 scripts/check_conflicts.py slots.json
  4 | slots.json: список слотов с полями slot_id, date (YYYY-MM-DD), start (HH:MM), duration_min.
  5 | Печатает пары slot_id, которые пересекаются по времени.
  6 | '''
  7 | import json
  8 | import sys
  9 | 
 10 | 
 11 | def interval(slot):
 12 |     '''Начало и конец слота.'''

## Задание 1. Таблица находок — 3 балла

Прочитайте пакет выше (номера строк напечатаны) и заполните `FINDINGS` — список словарей:

```python
{"file": "SKILL.md", "line": 12, "category": "...", "severity": "high", "problem": "...", "fix": "..."}
```

- `file` — путь внутри пакета (`SKILL.md`, `scripts/check_conflicts.py`, `references/teachers.md`);
- `line` — номер строки, где дефект виден; если дефект виден в нескольких строках, укажите одну;
- `category` — ключ из `CATEGORIES` (напечатаны ниже); `severity` ∈ `high | medium | low`;
- `problem` — что не так и к чему приведёт; `fix` — как исправить (по одному-два предложения, не короче 10 символов).

В пакете ровно **10 дефектов** (в ключе). Скрытая проверка сопоставляет находку с дефектом ключа по файлу, строке (допуск $\pm 2$)
и категории; каждый дефект засчитывается один раз. Баллы за число найденных $m$: $m \ge 9$ — 3; 8 — 2,5; 7 — 2; 6 — 1,5; 5 — 1; 4 — 0,5.
Лишние находки почти не штрафуются (−0,5, если их больше шести), но видны проверяющему.

Подсказка: смотрите не только на механические дефекты (их найдёт линтер), но и на смысл шагов: условие применимости,
полномочия, проверка результата, условие остановки, согласованность шагов, корректность скрипта.


In [4]:
for key, text in CATEGORIES.items():
    print(f"{key:18s} {text}")

FINDINGS = [
    {"file": "SKILL.md", "line": 3, "category": "trigger_scope", "severity": "medium",
     "problem": "Описание обещает помощь со всем курсом: навык выбирается для лекций и домашних заданий вместо записи.",
     "fix": "Указать поиск слотов, запись и отмену консультаций; обозначить неприменимость к учебным вопросам."},
    {"file": "SKILL.md", "line": 14, "category": "no_negative_scope", "severity": "medium",
     "problem": "Инструкция применять навык при любых сомнениях исключает границу неприменимости.",
     "fix": "Исключить общие вопросы о курсе, объяснения тем и расписание занятий; при неоднозначности уточнять намерение."},
    {"file": "SKILL.md", "line": 22, "category": "unknown_tool", "severity": "high",
     "problem": "consult_send_email отсутствует в каталоге: отправка подтверждения таким инструментом невозможна.",
     "fix": "Удалить вымышленный инструмент и обещание письма; сообщать подтверждённый результат в ответе."},
    {"file": "SKILL.md", "line": 24, "category": "secret", "severity": "high",
     "problem": "В инструкции опубликован токен, который предлагается передавать в каждый вызов без соответствующего параметра.",
     "fix": "Удалить секрет и указание его передавать; доступ настраивается вне пакета через разрешённый механизм окружения."},
    {"file": "SKILL.md", "line": 29, "category": "no_stop_condition", "severity": "high",
     "problem": "Повторять поиск до получения слотов означает бесконечную петлю при ошибке или отсутствии доступных мест.",
     "fix": "Задать максимум две попытки только при временном сбое; пустой результат и постоянная ошибка останавливают поиск."},
    {"file": "SKILL.md", "line": 32, "category": "contradiction", "severity": "high",
     "problem": "Шаг 5 требует немедленно бронировать первый слот, хотя шаг 4 требует дождаться выбора студента.",
     "fix": "Бронировать только явно выбранный и подтверждённый слот; без выбора остановиться и ждать ответа."},
    {"file": "SKILL.md", "line": 34, "category": "overreach", "severity": "high",
     "problem": "Для освобождения места предлагается отменить чужую активную бронь, что нарушает границы полномочий.",
     "fix": "Никогда не менять чужие брони; при заполнении слота предложить альтернативу с новым подтверждением."},
    {"file": "SKILL.md", "line": 35, "category": "no_verification", "severity": "high",
     "problem": "Сообщение об успешной записи не зависит от isError, наличия booking_id и статуса брони.",
     "fix": "До подтверждения успеха проверить isError=False, booking_id и status=active в фактическом ответе."},
    {"file": "SKILL.md", "line": 44, "category": "missing_file", "severity": "medium",
     "problem": "Ссылка на references/booking-policy.md не ведёт к существующему файлу, правила проверить невозможно.",
     "fix": "Убрать недействительную ссылку и не выдумывать политику; при необходимости запросить действующие правила."},
    {"file": "scripts/check_conflicts.py", "line": 13, "category": "script_bug", "severity": "high",
     "problem": "HHMM ошибочно используется как число минут; также <= считает соседние полуоткрытые интервалы пересечением.",
     "fix": "Перевести часы в минуты и использовать строгие сравнения для [start, start+duration), учитывая календарную дату."},
]

trigger_scope      description размыт, слишком широк или даёт ложные триггеры; не сказано, когда срабатывать
no_negative_scope  нет условия неприменимости (когда навык НЕ должен срабатывать)
unknown_tool       ссылка на инструмент, которого нет в TOOL_CATALOG, или неверное имя параметра
contradiction      шаги или разделы противоречат друг другу
no_stop_condition  нет условия остановки или лимита повторов
overreach          действие вне полномочий или без подтверждения пользователя
no_verification    нет проверки результата действия
secret             секрет, токен или ключ в тексте пакета
missing_file       ссылка на файл пакета, которого нет
script_bug         ошибка в коде scripts/
frontmatter        нет frontmatter, name или description; недопустимое name; description неразумной длины
bloat              тело SKILL.md раздуто: справочный материал должен лежать в references/


In [5]:
_lengths = {rel: len(text.splitlines()) for rel, text in PACKAGE.items()}
assert isinstance(FINDINGS, list) and len(FINDINGS) >= 8, "нужно не меньше 8 находок"
for item in FINDINGS:
    assert set(item) >= {"file", "line", "category", "severity", "problem", "fix"}, item
    assert item["file"] in PACKAGE, f"нет такого файла в пакете: {item['file']}"
    assert isinstance(item["line"], int) and 1 <= item["line"] <= _lengths[item["file"]], item
    assert item["category"] in CATEGORIES, item["category"]
    assert item["severity"] in SEVERITIES, item["severity"]
    assert len(item["problem"].strip()) >= 10 and len(item["fix"].strip()) >= 10, item
assert len({item["category"] for item in FINDINGS}) >= 5, "дефекты пакета относятся к разным категориям"
print("PASS: формат FINDINGS;", len(FINDINGS), "находок,", len({i['category'] for i in FINDINGS}), "категорий")


PASS: формат FINDINGS; 10 находок, 10 категорий


## Задание 2. Линтер навыка — 2 балла

Реализуйте `lint_skill(skill_dir, tool_catalog) -> list[dict]`. `skill_dir` — `pathlib.Path` папки навыка, `tool_catalog` — словарь
вида `TOOL_CATALOG`. Каждая находка: `{"file": "SKILL.md", "line": 21, "category": "...", "message": "..."}`;
`file` — путь относительно `skill_dir` через `/`, `line` — номер строки с 1, `category` — ключ из `CATEGORIES`.
Одна и та же тройка (файл, строка, категория) выдаётся один раз. Правила:

| Правило | Категория | Что проверять |
|---|---|---|
| R1 | `frontmatter` | `SKILL.md` есть и начинается с frontmatter; есть `name` и `description`; `name` соответствует `NAME_RE`, не длиннее 64 и равен имени папки; длина `description` от `MIN_DESCRIPTION` до `MAX_DESCRIPTION` символов. Строка — строка поля (или 1) |
| R2 | `unknown_tool` | в `SKILL.md` и `references/*.md` — слова вида `<префикс>_<буквы/цифры/_>`, где префикс — часть имени инструмента каталога до первого `_`; слова нет в каталоге |
| R3 | `missing_file` | в `SKILL.md` и `references/*.md` — пути, найденные `FILE_REF_RE`, которых нет в пакете |
| R4 | `secret` | во всех текстовых файлах пакета — строки, где срабатывает любой шаблон из `SECRET_PATTERNS` |
| R5 | `bloat` | тело `SKILL.md` (после frontmatter) длиннее `MAX_BODY_LINES` строк или `MAX_BODY_WORDS` слов; строка — первая строка тела |

Используйте готовые `parse_frontmatter`, `NAME_RE`, `FILE_REF_RE`, `SECRET_PATTERNS`. Линтер не должен падать на пакете без `references/`
и не должен давать ложных срабатываний на чистом пакете. Скрытые тесты: исходный пакет, скрытый чистый пакет (ноль находок)
и скрытый пакет с другими дефектами (все пять правил).

Механический линтер ловит не всё: смысловые дефекты из задания 1 остаются за человеком — это пригодится в отчёте.


In [6]:
def lint_skill(skill_dir, tool_catalog):
    """Пять механических правил; без привязки к исходным именам и строкам."""
    skill_dir = Path(skill_dir)
    findings = {}

    def add(file, line, category, message):
        key = (file, line, category)
        findings.setdefault(key, {"file": file, "line": line,
                                  "category": category, "message": message})

    texts = {}
    for path in sorted(skill_dir.rglob("*")):
        if not path.is_file():
            continue
        try:
            text = path.read_text(encoding="utf-8")
        except (UnicodeError, OSError):
            continue  # Бинарные ресурсы не являются текстом пакета.
        if "\x00" not in text:
            texts[path.relative_to(skill_dir).as_posix()] = text

    text = texts.get("SKILL.md")
    meta = parse_frontmatter(text) if text is not None else None
    if meta is None:
        add("SKILL.md", 1, "frontmatter", "Нужен SKILL.md с завершённым frontmatter в начале файла.")
    else:
        fields, lines = meta["fields"], meta["lines"]
        name = fields.get("name", "")
        if (not NAME_RE.fullmatch(name) or len(name) > 64
                or name != skill_dir.name):
            add("SKILL.md", lines.get("name", 1), "frontmatter",
                "name должен соответствовать формату, длине ≤64 и имени папки.")
        description = fields.get("description", "")
        if not MIN_DESCRIPTION <= len(description) <= MAX_DESCRIPTION:
            add("SKILL.md", lines.get("description", 1), "frontmatter",
                "description отсутствует или имеет недопустимую длину.")

    prefixes = sorted({name.split("_", 1)[0] for name in tool_catalog if "_" in name})
    tool_pattern = (re.compile(r"\b(?:" + "|".join(map(re.escape, prefixes))
                               + r")_[A-Za-z0-9_]+\b") if prefixes else None)
    for rel, content in texts.items():
        document = (rel == "SKILL.md" or
                    (rel.startswith("references/") and rel.endswith(".md")))
        for number, line in enumerate(content.splitlines(), 1):
            if document:
                if tool_pattern:
                    unknown = sorted(set(tool_pattern.findall(line)) - set(tool_catalog))
                    if unknown:
                        add(rel, number, "unknown_tool", "Нет в каталоге: " + ", ".join(unknown))
                for ref in set(FILE_REF_RE.findall(line)):
                    target = skill_dir / ref
                    if not target.is_file():
                        add(rel, number, "missing_file", "Отсутствует файл: " + ref)
            if any(pattern.search(line) for pattern in SECRET_PATTERNS):
                add(rel, number, "secret", "Строка похожа на секрет: удалите значение из пакета.")

    if text is not None:
        start = meta["body_start"] if meta else 1
        body_lines = text.splitlines()[start - 1:]
        if (len(body_lines) > MAX_BODY_LINES or
                len("\n".join(body_lines).split()) > MAX_BODY_WORDS):
            add("SKILL.md", start, "bloat", "Сократите тело; справочные сведения вынесите в references/.")
    return [findings[key] for key in sorted(findings)]

In [7]:
_found = lint_skill(SKILL_DIR, TOOL_CATALOG)
assert isinstance(_found, list) and all(isinstance(f, dict) for f in _found)
for f in _found:
    assert set(f) >= {"file", "line", "category", "message"} and f["category"] in CATEGORIES, f
assert {"unknown_tool", "secret", "missing_file"} <= {f["category"] for f in _found}, _found
assert lint_skill(CLEAN_DIR, TOOL_CATALOG) == [], "ложные срабатывания на чистом пакете"
_tmp = write_package(WORK / "no-frontmatter", {"SKILL.md": "# Без frontmatter\n"})
assert any(f["category"] == "frontmatter" and f["line"] == 1 for f in lint_skill(_tmp, TOOL_CATALOG))
for f in _found:
    print(f"{f['file']}:{f['line']}  {f['category']:13s} {f['message']}")
print("PASS: lint_skill (открытые проверки)")


SKILL.md:22  unknown_tool  Нет в каталоге: consult_send_email
SKILL.md:24  secret        Строка похожа на секрет: удалите значение из пакета.
SKILL.md:35  unknown_tool  Нет в каталоге: consult_send_email
SKILL.md:44  missing_file  Отсутствует файл: references/booking-policy.md
PASS: lint_skill (открытые проверки)


## Задание 3. Исправленный навык — 3 балла

Перепишите `SKILL.md` целиком в `FIXED_SKILL_MD` и исправьте скрипт в `FIXED_SCRIPT` (полный текст файла
`scripts/check_conflicts.py`). Если создаёте недостающие файлы, положите их в `FIXED_EXTRA_FILES = {"references/...": "текст"}`.

Требования:

- frontmatter: `name: consult-booking`; `description` говорит, что делает навык, когда его использовать и когда — нет;
- есть раздел о неприменимости; только инструменты из `TOOL_CATALOG` и только существующие файлы; никаких секретов;
- бронь и отмена — только после явного выбора/подтверждения студента и только своих броней;
- у повторов есть лимит и условие остановки; результат проверяется (`booking_id`, статус `active`);
- скрипт сохраняет интерфейс `overlaps(a, b) -> bool` и `find_conflicts(slots) -> list[tuple]` (отсортированные пары `slot_id`)
  и правильно считает пересечение интервалов $[s, s + d)$: соседние слоты не пересекаются.

**Автоматически (1 балл):** эталонный линтер не находит дефектов в исправленном пакете (0,4); скрипт проходит скрытые тесты
на пересечение интервалов (0,4); в тексте есть лимит повторов, подтверждение, проверка статуса `active` и условие неприменимости (0,2).
**Вручную (2 балла):** преподаватель читает `FIXED_SKILL_MD` и `FIXED_SCRIPT` целиком: устранены ли все 10 дефектов, не появилось ли новых,
осталось ли тело коротким и проверяемым.


In [8]:
FIXED_SKILL_MD = '''---
name: consult-booking
description: Ищет свободные слоты консультаций преподавателей, записывает студента на консультацию и отменяет его бронь. Используй для подбора времени консультации, бронирования выбранного слота, отмены записи и переноса через согласованные отмену и новую запись. Не используй для объяснения материала, решения домашних заданий, оценок, результатов экзаменов и расписания лекций.
---

# Запись на консультации

## Применимость и неприменимость
Применяй для поиска свободных консультаций, записи и отмены собственной брони студента.
Не применяй для общих вопросов о курсе, объяснения тем, решения заданий или расписания занятий.
Если намерение неоднозначно, сначала уточни его, не вызывая инструменты изменения данных.
Форматы преподавателей: `references/teachers.md`; актуальные место и доступность — только из карточки слота.

## Полномочия и входные данные
Уточни цель, student_id, тему, даты, формат и преподавателя при необходимости.
Не угадывай личность, booking_id или действующие правила отмены и лимиты броней.
Используй только consult_search_slots, consult_get_slot, consult_book, consult_cancel.
Чужие брони никогда не изменяй. Письма не отправляй: такого инструмента нет.

## Поиск и согласование
1. Вызови `consult_search_slots(date_from, date_to, format?, topic?, teacher?, limit=5, cursor?, response_format="concise")`.
2. Проверь isError и прочитай structuredContent либо документированный JSON из content.
   При пустом результате остановись, предложи изменить фильтры; не повторяй тот же поиск.
3. Если студент предоставил собственные действующие записи, сравни кандидатов с ними через
   `scripts/check_conflicts.py`: входной JSON содержит список объектов slot_id, date, start, duration_min.
   Пример запуска: `python3 scripts/check_conflicts.py slots.json`.
   Учитывай пары кандидат–существующая запись; конфликт кандидатов между собой не запрещает выбрать один.
   Если действующие записи неизвестны, попроси их или подтверждение отсутствия пересечений.
   Не утверждай, что проверил неизвестное расписание; в каталоге нет инструмента списка броней.
4. Покажи до трёх вариантов. Дождись явного выбора студента.
5. Получи выбранный слот через `consult_get_slot(slot_id)`, проверь актуальные время, формат и свободные места.
   Покажи точные параметры и запроси явное подтверждение бронирования. При изменении условий согласуй заново.

## Бронирование и проверка
1. Только после подтверждения вызови `consult_book(slot_id, student_id, topic, idempotency_key)`.
   Создай ключ один раз для этой попытки бронирования; при повторе используй тот же ключ и аргументы.
2. Сообщай «Вы записаны» только если isError=False, booking_id непустой и status равен `active`.
   Укажи подтверждённые идентификаторы, дату, время и место из полученной карточки.
3. При заполненном слоте не отменяй другие записи: предложи альтернативу и снова получи подтверждение.
   Ошибка, пустой или неполный ответ не подтверждают успех; сообщи неопределённость и остановись.

## Отмена и перенос
Попроси booking_id собственной брони и student_id, покажи отменяемую запись и дождись подтверждения.
Если принадлежность не установлена, остановись; не подставляй чужой student_id.
Вызови `consult_cancel(booking_id, student_id)`; успех — только isError=False и status=`cancelled`.
Перенос не атомарен: объясни риск потери места и отдельно согласуй отмену и новую запись.

## Лимит повторов и остановка
Для каждого логического действия разрешены максимум 2 попытки (1 первоначальная + 1 повтор).
Повтор допустим только при явно временной ошибке; неверные аргументы, отсутствие мест или пустой ответ — остановка.
При неопределённом исходе бронирования повторяй только с тем же idempotency_key и теми же аргументами.
После второй неудачи остановись и передай подсказку из ошибки; не выдумывай результат и не повторяй циклически.
Пагинация поиска — максимум 2 страницы с различными cursor; без cursor продолжение запрещено.
Без выбора, подтверждения или обязательных данных остановись и запроси недостающее у студента.
'''

FIXED_SCRIPT = '''"""Пересечения полуоткрытых интервалов консультаций, включая переход суток.

CLI: python3 scripts/check_conflicts.py slots.json
Вход — список slot_id, date (YYYY-MM-DD), start (HH:MM), duration_min.
"""
import json
import sys
from datetime import date, datetime


def interval(slot):
    """Начало и конец в минутах от полуночи; HHMM не является числом минут."""
    value = datetime.strptime(slot["start"], "%H:%M")
    duration = int(slot["duration_min"])
    if duration <= 0:
        raise ValueError("duration_min must be positive")
    start = value.hour * 60 + value.minute
    return start, start + duration


def overlaps(a, b):
    """Истина для непустого пересечения [start, end); соседство не пересечение."""
    a_start, a_end = interval(a)
    b_start, b_end = interval(b)
    a_offset = date.fromisoformat(a["date"]).toordinal() * 1440
    b_offset = date.fromisoformat(b["date"]).toordinal() * 1440
    return a_offset + a_start < b_offset + b_end and b_offset + b_start < a_offset + a_end


def find_conflicts(slots):
    """Отсортированные пары идентификаторов, каждый идентификатор пары упорядочен."""
    pairs = []
    for i, a in enumerate(slots):
        for b in slots[i + 1:]:
            if overlaps(a, b):
                pairs.append(tuple(sorted((a["slot_id"], b["slot_id"]))))
    return sorted(pairs)


if __name__ == "__main__":
    with open(sys.argv[1], encoding="utf-8") as stream:
        print(json.dumps(find_conflicts(json.load(stream)), ensure_ascii=False))
'''

FIXED_EXTRA_FILES = {}

In [9]:
assert FIXED_SKILL_MD.strip(), "FIXED_SKILL_MD пуст"
assert FIXED_SCRIPT.strip(), "FIXED_SCRIPT пуст"
_fixed_files = dict(PACKAGE, **{"SKILL.md": FIXED_SKILL_MD, "scripts/check_conflicts.py": FIXED_SCRIPT}, **FIXED_EXTRA_FILES)
FIXED_DIR = write_package(WORK / "fixed" / "consult-booking", _fixed_files)
_meta = parse_frontmatter(FIXED_SKILL_MD)
assert _meta and _meta["fields"].get("name") == "consult-booking", "frontmatter с name: consult-booking"
assert lint_skill(FIXED_DIR, TOOL_CATALOG) == [], lint_skill(FIXED_DIR, TOOL_CATALOG)

_ns = {"__name__": "check_conflicts"}
exec(compile(FIXED_SCRIPT, "check_conflicts.py", "exec"), _ns)
_a = {"slot_id": "s-1014-1", "date": "2026-10-14", "start": "10:00", "duration_min": 30}
_b = {"slot_id": "s-1014-2", "date": "2026-10-14", "start": "10:30", "duration_min": 15}
_c = {"slot_id": "s-1014-3", "date": "2026-10-14", "start": "10:20", "duration_min": 15}
assert _ns["overlaps"](_a, _b) is False, "соседние слоты не пересекаются"
assert _ns["overlaps"](_a, _c) is True
assert [tuple(p) for p in _ns["find_conflicts"]([_a, _b, _c])] == [("s-1014-1", "s-1014-3"), ("s-1014-2", "s-1014-3")]
print("PASS: исправленный пакет проходит ваш линтер, скрипт — открытые тесты")


PASS: исправленный пакет проходит ваш линтер, скрипт — открытые тесты


## Задание 4. Когда навык срабатывает — 2 балла

Агент выбирает навык по `description`. Настоящий выбор делает модель; здесь вместо неё **лексический прокси** `route(description, prompt)`:

- текст переводится в множество грубых основ (нижний регистр, без стоп-слов, срезано окончание, первые 5 букв);
- `description` делится на предложения; предложения со словами «не», «нельзя», «кроме» образуют отрицательную часть $N$, остальные — положительную $P$;
- для основ запроса $Q$ навык срабатывает, если $\frac{|Q \cap P|}{|Q|} \ge 0{,}25$ и $|Q \cap P| > |Q \cap N|$.

Это **не LLM** и не модель реального агента: прокси не понимает смысла и отрицаний, а только считает совпадения слов.
Его задача — дать воспроизводимую офлайн-метрику, чтобы сравнить старое и новое `description`, и показать, какие проверки нужны на настоящем агенте.

Заполните:

- `TRIGGER_SET` — **10** запросов `{"prompt": ..., "should_trigger": bool, "near_miss": bool}`: не меньше 4 должны срабатывать,
  не меньше 4 не должны, из них не меньше 2 «почти похожих» (`near_miss: True`) — запросы о курсе, преподавателях или времени,
  которые не про запись на консультацию; каждый запрос — не меньше 3 слов, без повторов;
- `REPORT` — три ответа (по 4–8 предложений, с числами из таблицы ниже):
  - `q1`: почему старое `description` плохо работает (какие запросы ложно срабатывают или пропускаются) и что изменили вы;
  - `q2`: разберите два почти похожих запроса: почему новое `description` их отсекает и где прокси ошибся бы там, где модель не ошибётся (и наоборот);
  - `q3`: как проверить срабатывание на настоящем агенте (набор, повторы, метрика) и какие дефекты задания 1 линтер не ловит в принципе.

**Автоматически (1 балл):** формат набора (0,25); почти похожие запросы есть и относятся к домену (0,25); новое `description`
лучше старого на вашем наборе по $F_1$ при recall $\ge 0{,}75$ (0,25); новое `description` на скрытом наборе запросов, $F_1 \ge 0{,}75$ (0,25).
**Вручную (1 балл):** ответы `REPORT`.


In [10]:
_STOP = set("""и в во на с со к ко по о об у за из для что как это мне я ты вы он она мы меня мой моя мою мои свой свою
можно ли а но или же бы до от при все всё так есть нужно надо хочу пожалуйста ещё уже там тут где когда какой какая какие
какую который этот эта эти тот то ну да нет его её их ему ей им наш ваш себя сам сама используй используйте применяй
навык the a an to of for and or is are""".split())
_NEG = {"не", "нельзя", "кроме"}
_ENDINGS = sorted("""иями ями ами ого его ому ему ыми ими ться тся ешь ишь ете ите ует уют ают яют ать ять ить еть ых их
ую юю ая яя ое ее ой ей ий ый ом ем ам ям ах ях ов ев ия ие ию ть у ю а я о е ы и ь й""".split(), key=len, reverse=True)


def _stem(word):
    for end in _ENDINGS:
        if word.endswith(end) and len(word) - len(end) >= 3:
            word = word[: -len(end)]
            break
    return word[:5]


def _words(text):
    return re.findall(r"[a-zа-я0-9]+", text.lower().replace("ё", "е"))


def stems(text):
    """Множество грубых основ: нижний регистр, без стоп-слов, срез окончания, первые 5 букв."""
    return {_stem(w) for w in _words(text) if w not in _STOP and w not in _NEG and len(w) > 1}


def split_description(description):
    """Основы «положительных» предложений и предложений с отрицанием (не/нельзя/кроме)."""
    positive, negative = set(), set()
    for sentence in re.split(r"[.;!?\n]+", description):
        target = negative if any(w in _NEG for w in _words(sentence)) else positive
        target |= stems(sentence)
    return positive, negative


def route_scores(description, prompt):
    positive, negative = split_description(description)
    p = stems(prompt)
    if not p:
        return 0.0, 0.0
    return len(p & positive) / len(p), len(p & negative) / len(p)


ROUTE_THRESHOLD = 0.25


def route(description, prompt, threshold=ROUTE_THRESHOLD):
    """ПРОКСИ роутера, не LLM: навык «срабатывает», если доля основ запроса, встречающихся
    в положительной части description, не меньше порога и больше доли из отрицательной части."""
    pos, neg = route_scores(description, prompt)
    return pos >= threshold and pos > neg


def trigger_metrics(description, trigger_set):
    tp = fp = fn = tn = 0
    for item in trigger_set:
        fired = route(description, item["prompt"])
        if item["should_trigger"]:
            tp += fired
            fn += not fired
        else:
            fp += fired
            tn += not fired
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)}


def description_of(skill_md):
    meta = parse_frontmatter(skill_md or "")
    return meta["fields"].get("description", "") if meta else ""


In [11]:
OLD_DESCRIPTION = description_of(PACKAGE["SKILL.md"])
for prompt in ["Запиши меня на консультацию в среду", "Когда следующая лекция курса?"]:
    pos, neg = route_scores(OLD_DESCRIPTION, prompt)
    print(f"{prompt!r}: P={pos:.2f} N={neg:.2f} -> {route(OLD_DESCRIPTION, prompt)}  основы={sorted(stems(prompt))}")


'Запиши меня на консультацию в среду': P=0.00 N=0.00 -> False  основы=['запиш', 'консу', 'сред']
'Когда следующая лекция курса?': P=0.33 N=0.00 -> True  основы=['курс', 'лекц', 'следу']


In [12]:
TRIGGER_SET = [{'prompt': 'Запиши меня на консультацию в среду', 'should_trigger': True, 'near_miss': False},
 {'prompt': 'Найди свободные слоты консультаций преподавателя',
  'should_trigger': True,
  'near_miss': False},
 {'prompt': 'Отмени мою бронь консультации', 'should_trigger': True, 'near_miss': False},
 {'prompt': 'Хочу перенести запись на консультацию', 'should_trigger': True, 'near_miss': False},
 {'prompt': 'Подбери время консультации по проекту', 'should_trigger': True, 'near_miss': False},
 {'prompt': 'Когда следующая лекция курса', 'should_trigger': False, 'near_miss': True},
 {'prompt': 'Какое расписание занятий преподавателя курса',
  'should_trigger': False,
  'near_miss': True},
 {'prompt': 'Объясни материал домашнего задания курса', 'should_trigger': False, 'near_miss': True},
 {'prompt': 'Покажи оценки за экзамен курса', 'should_trigger': False, 'near_miss': False},
 {'prompt': 'Какая погода завтра утром', 'should_trigger': False, 'near_miss': False}]

REPORT = {'q1': 'Старое description обещает помощь со всем курсом, но не называет запись, отмену и поиск '
       'слотов. На моих 10 запросах оно даёт TP=0, FP=2, FN=5, TN=3: precision=0.0, recall=0.0, '
       'F1=0.0. Например, вопрос о следующей лекции курса ложно включает навык, а просьба записать '
       'на консультацию пропускается. Я перечислил доменные действия и добавил отдельное '
       'предложение о неприменимости к лекциям, оценкам и решению заданий. Новое описание даёт '
       'TP=5, FP=0, FN=0, TN=5, precision=1.0, recall=1.0, F1=1.0. Это результат данного '
       'лексического прокси на составленном наборе, а не оценка качества LLM.',
 'q2': 'Два почти похожих запроса — «Когда следующая лекция курса» и «Какое расписание занятий '
       'преподавателя курса»: оба касаются курса и времени, но не записи на консультацию. Старое '
       'описание ложно включает первый по совпадению основы слова «курс», а второй исключает лишь '
       'потому, что доля совпадений ниже порога 0,25. Новое описание исключает расписание лекций и '
       'занятий, а совпадения положительной части для этих запросов недостаточны: обе строки '
       'получают False. В наборе всего 3 near_miss, все 3 новое описание отсекает, тогда как '
       'старое ложно включает 1 из 3; ещё одно ложное включение старого относится к оценкам за '
       'экзамен. Прокси может пропустить синоним «зарезервируй встречу с наставником», хотя '
       'реальная модель способна распознать намерение записи. Он также может включить «Не '
       'записывай меня на консультацию», потому что отрицание запроса исключается из основ, тогда '
       'как понимающий агент должен отказаться от действия. Обратное расхождение возможно, когда '
       'модель домысливает намерение записи по общему вопросу, хотя у прокси недостаточно '
       'совпадений; это гипотеза для отдельного теста, а не наблюдение LLM в ноутбуке.',
 'q3': 'Для проверки настоящего агента подготовлю отдельный размеченный набор из 40 запросов: 20 '
       'применимых и 20 неприменимых, включая отрицания, перефразирования и почти похожие '
       'намерения. Проведу по 3 запуска каждого запроса с одинаковыми доступными инструментами, '
       'сохраняя решение выбора навыка и фактические действия. Рассчитаю precision, recall, F1 и '
       'долю неподтверждённых изменений; 95%-е интервалы получу бутстрэпом целых запросов, '
       'сохраняя три повтора. Текущие F1=1.0 и recall=1.0 получены лишь на 10 строках прокси и не '
       'заменяют такой проверки. Механический линтер реализует 5 правил и не доказывает '
       'адекватность триггера, неприменимость, согласованность шагов, соблюдение полномочий, '
       'остановку повторов или проверку успеха. Ошибку арифметики интервалов он тоже не понимает: '
       'нужны поведенческие тесты соседства, смены часа, вложенности и календарных дат. Скрытые '
       'чистые и дефектные пакеты нужны для оценки обобщения линтера, а не подгонки под известные '
       'номера строк.'}

In [13]:
assert isinstance(TRIGGER_SET, list) and len(TRIGGER_SET) == 10, "нужно ровно 10 запросов"
assert sum(bool(t["should_trigger"]) for t in TRIGGER_SET) >= 4
assert sum(not t["should_trigger"] for t in TRIGGER_SET) >= 4
assert sum(bool(t.get("near_miss")) and not t["should_trigger"] for t in TRIGGER_SET) >= 2
assert len({t["prompt"].strip().lower() for t in TRIGGER_SET}) == 10, "запросы повторяются"
assert all(len(t["prompt"].split()) >= 3 for t in TRIGGER_SET)
assert all(isinstance(REPORT.get(k), str) and len(REPORT[k].strip()) >= 200 for k in ("q1", "q2", "q3")), "REPORT: q1–q3 по 4–8 предложений"

NEW_DESCRIPTION = description_of(FIXED_SKILL_MD)
print(f"{'запрос':60s} {'надо':>5s} {'старое':>7s} {'новое':>6s}")
for t in TRIGGER_SET:
    print(f"{t['prompt'][:60]:60s} {str(t['should_trigger']):>5s} {str(route(OLD_DESCRIPTION, t['prompt'])):>7s} "
          f"{str(route(NEW_DESCRIPTION, t['prompt'])):>6s}")
METRICS = {"old": trigger_metrics(OLD_DESCRIPTION, TRIGGER_SET), "new": trigger_metrics(NEW_DESCRIPTION, TRIGGER_SET)}
print(json.dumps(METRICS, ensure_ascii=False, indent=1))
assert METRICS["new"]["f1"] > METRICS["old"]["f1"], "новое description должно быть лучше старого на вашем наборе"
print("PASS: TRIGGER_SET и REPORT (открытые проверки)")


запрос                                                        надо  старое  новое
Запиши меня на консультацию в среду                           True   False   True
Найди свободные слоты консультаций преподавателя              True   False   True
Отмени мою бронь консультации                                 True   False   True
Хочу перенести запись на консультацию                         True   False   True
Подбери время консультации по проекту                         True   False   True
Когда следующая лекция курса                                 False    True  False
Какое расписание занятий преподавателя курса                 False   False  False
Объясни материал домашнего задания курса                     False   False  False
Покажи оценки за экзамен курса                               False    True  False
Какая погода завтра утром                                    False   False  False
{
 "old": {
  "tp": 0,
  "fp": 2,
  "fn": 5,
  "tn": 3,
  "precision": 0.0,
  "recall": 0.0,
  "f1

## Перед сдачей

1. *Kernel → Restart & Run All*: ноутбук выполняется сверху вниз, все открытые проверки печатают `PASS`.
2. Заполнен `STUDENT` (ФИО и группа).
3. `FINDINGS`, `lint_skill`, `FIXED_SKILL_MD`, `FIXED_SCRIPT`, `TRIGGER_SET`, `REPORT` заполнены и не переименованы.
4. В `REPORT` — ваши выводы с числами из таблицы задания 4; про качество LLM выводов нет (её здесь не вызывали).
5. Сохраните ноутбук и загрузите файл `.ipynb` в Мирер.
